# Advanced Wave Dynamics

[Full course sequence](../../ai4sci/README.md) · [Start notebook](../../Start_Here.ipynb)

All levels are retained and use PhysicsNeMo **2.2.2** `FullyConnected`, SymPy `PDE`, `PhysicsInformer`, and explicit PyTorch training loops. Complete `student_equations` in the linked `.py` file. Until that function is completed, the default run stops with an explanatory error. If you get stuck, set `USE_REFERENCE=True` to run and compare the completed implementation.

**Learning workflow:** inspect the equations and conditions → edit and save the linked `.py` file → run it → inspect the PDE and condition errors and the predictions. A successful short run does not establish convergence. Each run writes to a new directory.

`PhysicsInformer` computes spatial derivatives from `coordinates`. For the current API, the training code computes time derivatives with PyTorch autograd and supplies keys such as `u__t` and `u__t__t`. Read `loss_terms` and the explicit `optimizer.zero_grad → backward → step` loop in each training file.

Instructor automation can execute these same cells with the environment variables `AI4SCI_REFERENCE=1`, `AI4SCI_DEVICE=cpu`, and `AI4SCI_STEPS=2`. The default remains the student exercise mode.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "challenge" / "wave" / "wave_l1.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the repository.")
LAB_DIR = ROOT / "challenge" / "wave"
USE_REFERENCE = os.environ.get("AI4SCI_REFERENCE", "0").lower() in {"1", "true", "yes"}  # Default: student exercise mode.
DEVICE = os.environ.get("AI4SCI_DEVICE", "auto")
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))  # Verify runtime and accuracy on the event GPU.
SEED = 42
RUN_TAG = datetime.now().strftime("%Y%m%d-%H%M%S-%f")


## Level 1 · Basic 2D wave equation

The spatial domain is $[0,\pi]^2$, the time interval is $[0,2\pi]$, and $c=1$.
$$u_{tt}-c^2(u_{xx}+u_{yy})=0$$
Both the initial displacement and initial velocity are $\sin x\sin y$. The displacement is zero on all four edges.
$$u=\sin x\sin y[\cos(\sqrt2t)+\sin(\sqrt2t)/\sqrt2]$$
This corrected analytic solution preserves the original initial conditions. `exact_reference` is used only for validation. Differentiate the analytic solution to verify the PDE and both initial conditions.

### Code and exercise

Open [wave_l1.py](wave_l1.py) and inspect `reference_equations`, `student_equations`, `loss_terms`, and `main`. Write the required dictionary of PDE residuals in `student_equations`, then save with **Ctrl+S / ⌘S**. Identify where the applicable initial, boundary, and integral conditions enter the loss.

Start with a small run equivalent to `--steps 2 --device cpu --reference` to check the complete input/output path. The student and reference implementations share the same sampling and evaluation code.


In [ ]:
RUN_COMPLETED = False
result_dir = LAB_DIR / "outputs" / f"wave_l1-{RUN_TAG}"
command = [sys.executable, str(LAB_DIR / "wave_l1.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED = True


### Inspect the actual results

The first and last held-out rows in `loss.csv` are evaluated at **identical points**. Intermediate rows describe freshly sampled training minibatches. Distinguish the initial/final errors from the PDE and condition residuals in `metrics.json`. A reference error is included only when a comparable reference is available. `model.pt` stores the model state and configuration; `predictions.npz` contains the actual prediction arrays.


In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError("The current training run has not completed.")
metrics = json.loads((result_dir / "metrics.json").read_text())
assert metrics["steps"] == STEPS and metrics["seed"] == SEED
print(json.dumps(metrics, indent=2))
preview = result_dir / "preview.png"
if preview.is_file():
    from IPython.display import display, Image
    display(Image(filename=str(preview)))
else:
    print("Plotting dependencies are unavailable. Inspect the actual arrays in predictions.npz.")


## Level 2 · Variable Wave Speed

The spatial domain and time interval are the same as in Level 1. The speed is $c(x,y)=1+0.5\sin x\cos y$, and the equation is
$$u_{tt}-c(x,y)^2(u_{xx}+u_{yy})=0.$$
This exercise uses the non-divergence form above. Do not replace it with $\nabla\cdot(c^2\nabla u)$.
The initial displacement is $\sin x\sin y$, the initial velocity is zero, and the displacement is zero on all four edges. No verified closed-form reference is provided, so inspect the PDE, initial-condition, and boundary-condition residuals separately. Reduced residuals alone do not establish solution accuracy over the entire domain and time interval.

### Code and exercise

Open [wave_l2.py](wave_l2.py) and inspect `reference_equations`, `student_equations`, `loss_terms`, and `main`. Write the required dictionary of PDE residuals in `student_equations`, then save with **Ctrl+S / ⌘S**. Identify where the applicable initial, boundary, and integral conditions enter the loss.

Start with a small run equivalent to `--steps 2 --device cpu --reference` to check the complete input/output path. The student and reference implementations share the same sampling and evaluation code.


In [ ]:
RUN_COMPLETED = False
result_dir = LAB_DIR / "outputs" / f"wave_l2-{RUN_TAG}"
command = [sys.executable, str(LAB_DIR / "wave_l2.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED = True


### Inspect the actual results

The first and last held-out rows in `loss.csv` are evaluated at **identical points**. Intermediate rows describe freshly sampled training minibatches. Distinguish the initial/final errors from the PDE and condition residuals in `metrics.json`. A reference error is included only when a comparable reference is available. `model.pt` stores the model state and configuration; `predictions.npz` contains the actual prediction arrays.


In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError("The current training run has not completed.")
metrics = json.loads((result_dir / "metrics.json").read_text())
assert metrics["steps"] == STEPS and metrics["seed"] == SEED
print(json.dumps(metrics, indent=2))
preview = result_dir / "preview.png"
if preview.is_file():
    from IPython.display import display, Image
    display(Image(filename=str(preview)))
else:
    print("Plotting dependencies are unavailable. Inspect the actual arrays in predictions.npz.")


## Level 3 · Complex Boundaries and Circular Domain

The spatial domain is the interior of a circle of radius 1, the time interval is $[0,3]$, and $c=1$.
$$u_{tt}-\Delta u=0,\qquad u+0.5\partial_nu=0\quad\text{on }r=1.$$
$$u(x,y,0)=e^{-20((x-.3)^2+y^2)}+e^{-20((x+.3)^2+y^2)},\quad u_t(x,y,0)=0.$$
The outward unit normal on the circle is $(x,y)$, so `boundary_residual` is $u+0.5(xu_x+yu_y)$. The two Gaussian sources and the Robin condition are preserved from the original exercise. The initial function does not exactly satisfy the Robin condition at the boundary, so also inspect errors near the boundary at the initial time. No closed-form reference is claimed.

### Code and exercise

Open [wave_l3.py](wave_l3.py) and inspect `reference_equations`, `student_equations`, `loss_terms`, and `main`. Write the required dictionary of PDE residuals in `student_equations`, then save with **Ctrl+S / ⌘S**. Identify where the applicable initial, boundary, and integral conditions enter the loss.

Start with a small run equivalent to `--steps 2 --device cpu --reference` to check the complete input/output path. The student and reference implementations share the same sampling and evaluation code.


In [ ]:
RUN_COMPLETED = False
result_dir = LAB_DIR / "outputs" / f"wave_l3-{RUN_TAG}"
command = [sys.executable, str(LAB_DIR / "wave_l3.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED = True


### Inspect the actual results

The first and last held-out rows in `loss.csv` are evaluated at **identical points**. Intermediate rows describe freshly sampled training minibatches. Distinguish the initial/final errors from the PDE and condition residuals in `metrics.json`. A reference error is included only when a comparable reference is available. `model.pt` stores the model state and configuration; `predictions.npz` contains the actual prediction arrays.


In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError("The current training run has not completed.")
metrics = json.loads((result_dir / "metrics.json").read_text())
assert metrics["steps"] == STEPS and metrics["seed"] == SEED
print(json.dumps(metrics, indent=2))
preview = result_dir / "preview.png"
if preview.is_file():
    from IPython.display import display, Image
    display(Image(filename=str(preview)))
else:
    print("Plotting dependencies are unavailable. Inspect the actual arrays in predictions.npz.")


## Check your understanding and continue

- How do the inputs, outputs, equations, and initial/boundary conditions change between levels?
- Are training minibatch loss and error at fixed validation points the same metric?
- Does the problem have an independent reference? If so, do its assumptions match the current configuration?
- Compare changes to sample counts, training steps, and condition weights using new run directories and saved configurations.

[Next challenge](../fuild/Fluid_Structure_Interaction.ipynb) · [Full course sequence](../../ai4sci/README.md)

Adapted from the original OpenHackathons materials, with the file-specific copyright notices retained. [License](../../LICENSE).


--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.